cicli innestati
            - Nei cicli innestati, in OpenMP si parallelizza di solito solo il ciclo più esterno (`#pragma omp parallel for`): le iterazioni esterne vengono distribuite tra i thread, mentre quelle interne restano sequenziali per ciascun thread.  
            - Esempio con 4 thread: ogni thread riceve diversi valori di `i` (ciclo esterno) e, per ciascuno, esegue tutti i valori di `j` (ciclo interno).

- **Soluzione corretta: collasso dei cicli.**  
    Due cicli annidati di dimensioni `n` e `m` vengono trasformati in un unico ciclo di lunghezza `n * m`, ricostruendo gli indici originali con divisione intera e modulo.  
- In OpenMP questo si fa direttamente con `#pragma omp for collapse(2)`, che unisce automaticamente i due cicli più esterni successivi.


In [4]:
#if defined(_OPENMP)
  #include <omp.h>
#endif
#include <stdio.h>
#include <stdlib.h>

int main(int argc, char *argv[]) {
    const int n = 100000;
    int *v = (int *)malloc(n * sizeof(int));
    if (v == NULL) {
        fprintf(stderr, "malloc fallita\n");
        return 1;
    }

    // Questa dipendenza (v[i-10]) rende il ciclo non parallelizzabile in sicurezza.
    for (int i = 0; i < n; i++) {
        if (i > 10) {
            v[i] = i + v[i - 10];
        } else {
            v[i] = i;
        }
    }

    int sum = 0;
#pragma omp parallel for reduction(+ : sum)
    for (int i = 0; i < n; i++) {
        sum += v[i];
    }

    printf("sum = %d\n", sum);
    free(v);
    return 0;
}

sum = 148649224


task : come paralizzarli (#pragma omp task)
    -ho funzioni eseguire parallelo, possono messi coda, creo pool thread eseguire, serve thread dispensa compiti

Chiamare C da Python:
    -librerie condivise -> convertire tipi python in tipi C, specificare tipi ritorno
    -problemi: tipo argomenti, tipo ritorno, chi gestisce memoria
       - int py -> precisione arbitraria, non fissata, C -> 32/64 bit
       - char c -> ASCII, py -> stringa unicode
    -![alt text](image-1.png)
    -specificare tipo argomenti:
        - Ricordate che è possibile compilare un file come libreria condivisa con le opzioni -fPIC e -shared del compilatore:
          - `gcc -fPIC -shared -o library.so library.c`
        - Servirà poi caricare la libreria da Python usando ctypes:
          - `ctypes.cdll.LoadLibrary("library.so")`
